This notebook demonstrates my winning solution for the Santa 2025 competition. See also the writeup here: XXX

To allow it to run in the Kaggle environment, it runs only a few tree combinations, covering the various solution flavors used. The full winning submission involved more steps, many of them manual:
- Rerun each number of trees several times for different configurations (symmetric vs asymmetric, tesselated vs free) and seeds.
- An additional 'exploit' step in which the best solutions from multiple runs for the same number of trees are combined in their own genetic algorithm.
- An additional compression step performed in float64 (instead of float32 as this notebook does) to squeeze out some decimal dust.
- Using a few manually built crystal solutions (some of them taken from public notebooks).

This means that the current notebook may not reproduce scores as good as my submission, particularly for high numbers of trees.

If you do want to run this for real, the most cost- and energy-efficient GPU to use is an RTX 5090. Including multiprocessing tricks, it could run the solvers in this notebook in well under an hour.

In [ ]:
# Set up path and load core modules
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), 'core'))
import kaggle_support as kgs
import pack_ga3
import pack_io
import pandas as pd
import pack_vis_sol
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

In [ ]:
# Read in the best solutions - this will be used in modes 4 and 5 only (see "solve" below)
# I used these modes for odd numbers of trees - they use the solutions already found for even numbers of trees
best_solutions,_ = pack_io.dataframe_to_solution_list(pd.read_csv(kgs.submission_csv_path))

# Define the solver
def solve(num_trees:int, mode:int, seed:int=42) -> kgs.SolutionCollection:
    """Create and configure a GA runner for packing `num_trees`."""
    """
    This helper selects and configures a `pack_ga3` runner according to
    the `mode` argument, optionally seeding it from previously saved
    solutions. 

    Parameters:
    - num_trees (int): number of trees to pack.
    - mode (int): solver mode selector. Allowed values:
        0 : asymmetric
        1 : 180 degrees symmetric
        2 : 180 degrees symmetric, tesselated
        3 : 180 degrees symmetric, tesselated with half-cell offset
        4 : asymmetric, seed from N-1 solution
        5 : asymmetric, seed from N+1 solution
        6 : 90 degrees symmetric
    - seed (int): RNG seed for the genetic algorithm runner.

    Returns the packed configuration (as a kgs.SolutionCollection).
    """
    # Mode 0: asymmetric
    # Mode 1: 180 degrees symmetric
    # Mode 2: 180 degrees symmetric, tesselated
    # Mode 3: 180 degrees symmetric, tesselated with half-cell offset
    # Mode 4: asymmetric, use N-1 solution as seed
    # Mode 5: asymmetric, use N+1 solution as seed
    # Mode 6: 90 degrees symmetric - not used in my submission, but makes some cool solutions    

    match mode:
        case 0:
            runner = pack_ga3.baseline()
        case 1:
            runner = pack_ga3.baseline_symmetry_180()
        case 2:
            runner = pack_ga3.baseline_symmetry_180_tesselated()
            # Set the Y position of the crystal
            runner.ga.ga_base.initializer.ref_sol_axis2_offset = lambda r:0.
        case 3:
            runner = pack_ga3.baseline_symmetry_180_tesselated()
            # Set the Y position of the crystal, offset by half a cell
            runner.ga.ga_base.initializer.ref_sol_axis2_offset = lambda r:0.5
        case 4:
            runner = pack_ga3.baseline_tesselated()       
            runner.ga.ga_base.initializer.ref_sol_crystal_type = None   
            # Set seed solutions; the inner part will be used, effectively seeding the crystal
            runner.ga.ga_base.initializer.ref_sol = best_solutions[num_trees-2]
        case 5:
            runner = pack_ga3.baseline_tesselated()       
            runner.ga.ga_base.initializer.ref_sol_crystal_type = None     
            # Set seed solutions; the inner part will be used, effectively seeding the crystal
            runner.ga.ga_base.initializer.ref_sol = best_solutions[num_trees]
        case 6:
            runner = pack_ga3.baseline_symmetry_90()
        case _:
            raise ValueError("Invalid mode")
        
    runner.ga.ga_base.N_trees_to_do = num_trees
    runner.diagnostic_plot = True  # Show diagnostic plots
    runner.seed = seed   # Set random seed    
    runner.plot_every = 20 # Diagnostic plots every 20 iterations
    runner.ga.stop_check_generations_scale = 1 # Stop sooner if no improvement (just to speed up this notebook)
    #runner.n_generations = 5


    # Off we go!
    runner.run()

    # Show the final diagnostic plot    
    clear_output(wait=True)  # Clear previous output          
    runner.ga.diagnostic_plots(runner._current_generation, None)  
    plt.close('all')

    # Show the solution, and compare it to the submitted one
    _,ax = plt.subplots(1,2,figsize=(12,6))
    plt.sca(ax[0])
    pack_vis_sol.pack_vis_sol(runner.ga.champions[0].phenotype, ax=ax[0])
    plt.title(f'Solution generated just now, score={runner.ga.champions[0].phenotype.h[0,0].get()**2/num_trees:.6f}')
    plt.sca(ax[1])
    pack_vis_sol.pack_vis_sol(best_solutions[num_trees-1], ax=ax[1])
    plt.title(f'Solution from winning submission, score={best_solutions[num_trees-1].h[0,0].get()**2/num_trees:.6f}')


In [ ]:
solve(7, 0)